# Step 2 — Skill Dictionary & Normalisation

**Project:** From Australian Study Materials to U.S. Career Opportunities  
**Purpose:** Build a reproducible skill taxonomy and normalisation layer for U.S. Data/AI job advertisements.

This notebook takes the cleaned job dataset produced in Step 1 and:

1. Defines a controlled technical skill taxonomy.
2. Maps aliases to canonical skill names.
3. Applies safe phrase/token matching rules to job descriptions.
4. Computes overall and career-level skill frequencies.
5. Exports files for the Step 3 spaCy `PhraseMatcher` pipeline.

> Skill frequencies here describe mentions in job advertisements. They do not measure a candidate's mastery, employability, or probability of obtaining a job.

In [4]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_rows", 150)
pd.set_option("display.max_columns", 20)

CANDIDATE_PATHS = [
    Path("job_cleaned.csv"),
    Path("data/job_cleaned.csv"),
    Path("/mnt/data/job_cleaned.csv"),
]

JOBS_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if JOBS_PATH is None:
    raise FileNotFoundError(
        "job_cleaned.csv was not found. Put it in the same folder as this notebook "
        "or update CANDIDATE_PATHS."
    )

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Input:", JOBS_PATH)
print("Output folder:", OUTPUT_DIR.resolve())

Input: job_cleaned.csv
Output folder: E:\数据分析\36118\asm2\outputs


## 1. Load and validate the cleaned job data

Step 1 is responsible for data cleaning. This notebook therefore treats `job_cleaned.csv`
as the input and performs only the validation needed for skill-dictionary construction.

In [6]:
jobs = pd.read_csv(JOBS_PATH)

required_columns = {"job_id", "career_category", "job_title", "job_description"}
missing = required_columns - set(jobs.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print("Number of job advertisements:", len(jobs))
print("\nCareer categories:")
print(jobs["career_category"].value_counts(dropna=False))

print("\nMissing job descriptions:", jobs["job_description"].isna().sum())
print("Duplicate job IDs:", jobs["job_id"].duplicated().sum())

Number of job advertisements: 688

Career categories:
career_category
AI Engineer       275
ML Engineer       202
Data Scientist    127
Data Engineer      57
Data Analyst       27
Name: count, dtype: int64

Missing job descriptions: 0
Duplicate job IDs: 0


## 2. Define the controlled skill taxonomy

The dictionary uses three fields:

- `canonical_skill`: the standard skill name used by the system.
- `alias`: a spelling, abbreviation, or alternative expression found in job advertisements.
- `category`: a higher-level skill group.

Examples:

- `ML` → `Machine Learning`
- `LLM` / `LLMs` → `Large Language Models`
- `Amazon Web Services` → `AWS`
- `PowerBI` → `Power BI`

The taxonomy is controlled rather than generated from every frequent word. This reduces noise and keeps later matching explainable.

In [8]:
skill_records = [{'canonical_skill': 'Python', 'alias': 'python', 'category': 'Programming'}, {'canonical_skill': 'SQL', 'alias': 'sql', 'category': 'Programming'}, {'canonical_skill': 'R', 'alias': 'R', 'category': 'Programming'}, {'canonical_skill': 'R', 'alias': 'r programming', 'category': 'Programming'}, {'canonical_skill': 'R', 'alias': 'r language', 'category': 'Programming'}, {'canonical_skill': 'Java', 'alias': 'java', 'category': 'Programming'}, {'canonical_skill': 'Scala', 'alias': 'scala', 'category': 'Programming'}, {'canonical_skill': 'C++', 'alias': 'c++', 'category': 'Programming'}, {'canonical_skill': 'C++', 'alias': 'cpp', 'category': 'Programming'}, {'canonical_skill': 'C#', 'alias': 'c#', 'category': 'Programming'}, {'canonical_skill': 'C#', 'alias': 'c sharp', 'category': 'Programming'}, {'canonical_skill': 'JavaScript', 'alias': 'javascript', 'category': 'Programming'}, {'canonical_skill': 'JavaScript', 'alias': 'java script', 'category': 'Programming'}, {'canonical_skill': 'TypeScript', 'alias': 'typescript', 'category': 'Programming'}, {'canonical_skill': 'TypeScript', 'alias': 'type script', 'category': 'Programming'}, {'canonical_skill': 'Go', 'alias': 'golang', 'category': 'Programming'}, {'canonical_skill': 'Bash', 'alias': 'bash', 'category': 'Programming'}, {'canonical_skill': 'Bash', 'alias': 'shell scripting', 'category': 'Programming'}, {'canonical_skill': 'MATLAB', 'alias': 'matlab', 'category': 'Programming'}, {'canonical_skill': 'Statistics', 'alias': 'statistics', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Statistics', 'alias': 'statistical analysis', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Statistics', 'alias': 'statistical modeling', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Statistics', 'alias': 'statistical modelling', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'A/B Testing', 'alias': 'a/b testing', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'A/B Testing', 'alias': 'a/b test', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'A/B Testing', 'alias': 'ab testing', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Hypothesis Testing', 'alias': 'hypothesis testing', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Experiment Design', 'alias': 'experiment design', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Experiment Design', 'alias': 'experimental design', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Time Series', 'alias': 'time series', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Time Series', 'alias': 'time-series', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Forecasting', 'alias': 'forecasting', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Regression', 'alias': 'regression', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Classification', 'alias': 'classification', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Clustering', 'alias': 'clustering', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Causal Inference', 'alias': 'causal inference', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Data Analysis', 'alias': 'data analysis', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Data Cleaning', 'alias': 'data cleaning', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Data Cleaning', 'alias': 'data cleansing', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Feature Engineering', 'alias': 'feature engineering', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'EDA', 'alias': 'exploratory data analysis', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'EDA', 'alias': 'eda', 'category': 'Data Analysis & Statistics'}, {'canonical_skill': 'Machine Learning', 'alias': 'machine learning', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Machine Learning', 'alias': 'ml models', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Machine Learning', 'alias': 'ml model', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Deep Learning', 'alias': 'deep learning', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Computer Vision', 'alias': 'computer vision', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Recommendation Systems', 'alias': 'recommendation systems', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Recommendation Systems', 'alias': 'recommender systems', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Recommendation Systems', 'alias': 'recommendation system', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Recommendation Systems', 'alias': 'recommender system', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Reinforcement Learning', 'alias': 'reinforcement learning', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Generative AI', 'alias': 'generative ai', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Generative AI', 'alias': 'genai', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Generative AI', 'alias': 'gen ai', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Artificial Intelligence', 'alias': 'artificial intelligence', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Predictive Modeling', 'alias': 'predictive modeling', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Predictive Modeling', 'alias': 'predictive modelling', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Model Evaluation', 'alias': 'model evaluation', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Model Evaluation', 'alias': 'model validation', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Model Training', 'alias': 'model training', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Model Training', 'alias': 'train models', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Model Training', 'alias': 'training models', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Feature Store', 'alias': 'feature store', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Feature Store', 'alias': 'feature stores', 'category': 'Machine Learning & AI'}, {'canonical_skill': 'Natural Language Processing', 'alias': 'natural language processing', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Natural Language Processing', 'alias': 'nlp', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Large Language Models', 'alias': 'large language models', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Large Language Models', 'alias': 'large language model', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Large Language Models', 'alias': 'llms', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Large Language Models', 'alias': 'llm', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Transformers', 'alias': 'transformer models', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Transformers', 'alias': 'transformer model', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Transformers', 'alias': 'transformers', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'RAG', 'alias': 'retrieval augmented generation', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'RAG', 'alias': 'retrieval-augmented generation', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'RAG', 'alias': 'rag pipeline', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'RAG', 'alias': 'rag systems', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'RAG', 'alias': 'rag system', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Prompt Engineering', 'alias': 'prompt engineering', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Prompt Engineering', 'alias': 'prompt design', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Embeddings', 'alias': 'embeddings', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Embeddings', 'alias': 'embedding models', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Embeddings', 'alias': 'embedding model', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Vector Search', 'alias': 'vector search', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Vector Search', 'alias': 'semantic search', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Hugging Face', 'alias': 'hugging face', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Hugging Face', 'alias': 'huggingface', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'LangChain', 'alias': 'langchain', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'LlamaIndex', 'alias': 'llamaindex', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'LlamaIndex', 'alias': 'llama index', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Fine-tuning', 'alias': 'fine-tuning', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Fine-tuning', 'alias': 'fine tuning', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Fine-tuning', 'alias': 'finetuning', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Tokenization', 'alias': 'tokenization', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Tokenization', 'alias': 'tokenisation', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Named Entity Recognition', 'alias': 'named entity recognition', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Named Entity Recognition', 'alias': 'ner', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'Text Classification', 'alias': 'text classification', 'category': 'NLP / Generative AI'}, {'canonical_skill': 'PyTorch', 'alias': 'pytorch', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'TensorFlow', 'alias': 'tensorflow', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'scikit-learn', 'alias': 'scikit-learn', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'scikit-learn', 'alias': 'scikit learn', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'scikit-learn', 'alias': 'sklearn', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'XGBoost', 'alias': 'xgboost', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'LightGBM', 'alias': 'lightgbm', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'Keras', 'alias': 'keras', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'Pandas', 'alias': 'pandas', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'NumPy', 'alias': 'numpy', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'SciPy', 'alias': 'scipy', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'spaCy', 'alias': 'spacy', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'NLTK', 'alias': 'nltk', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'JAX', 'alias': 'jax', 'category': 'ML Frameworks & Libraries'}, {'canonical_skill': 'Apache Spark', 'alias': 'apache spark', 'category': 'Data Engineering'}, {'canonical_skill': 'Apache Spark', 'alias': 'spark', 'category': 'Data Engineering'}, {'canonical_skill': 'Kafka', 'alias': 'apache kafka', 'category': 'Data Engineering'}, {'canonical_skill': 'Kafka', 'alias': 'kafka', 'category': 'Data Engineering'}, {'canonical_skill': 'Airflow', 'alias': 'apache airflow', 'category': 'Data Engineering'}, {'canonical_skill': 'Airflow', 'alias': 'airflow', 'category': 'Data Engineering'}, {'canonical_skill': 'Hadoop', 'alias': 'hadoop', 'category': 'Data Engineering'}, {'canonical_skill': 'dbt', 'alias': 'dbt', 'category': 'Data Engineering'}, {'canonical_skill': 'ETL', 'alias': 'etl', 'category': 'Data Engineering'}, {'canonical_skill': 'ETL', 'alias': 'extract transform load', 'category': 'Data Engineering'}, {'canonical_skill': 'ETL', 'alias': 'extract, transform, load', 'category': 'Data Engineering'}, {'canonical_skill': 'ELT', 'alias': 'elt', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Pipelines', 'alias': 'data pipelines', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Pipelines', 'alias': 'data pipeline', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Warehousing', 'alias': 'data warehouse', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Warehousing', 'alias': 'data warehousing', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Warehousing', 'alias': 'data warehouses', 'category': 'Data Engineering'}, {'canonical_skill': 'Databricks', 'alias': 'databricks', 'category': 'Data Engineering'}, {'canonical_skill': 'Snowflake', 'alias': 'snowflake', 'category': 'Data Engineering'}, {'canonical_skill': 'Flink', 'alias': 'apache flink', 'category': 'Data Engineering'}, {'canonical_skill': 'Flink', 'alias': 'flink', 'category': 'Data Engineering'}, {'canonical_skill': 'Hive', 'alias': 'apache hive', 'category': 'Data Engineering'}, {'canonical_skill': 'Hive', 'alias': 'hive', 'category': 'Data Engineering'}, {'canonical_skill': 'Presto', 'alias': 'presto', 'category': 'Data Engineering'}, {'canonical_skill': 'Trino', 'alias': 'trino', 'category': 'Data Engineering'}, {'canonical_skill': 'Lakehouse', 'alias': 'lakehouse', 'category': 'Data Engineering'}, {'canonical_skill': 'Lakehouse', 'alias': 'data lakehouse', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Lake', 'alias': 'data lake', 'category': 'Data Engineering'}, {'canonical_skill': 'Data Lake', 'alias': 'data lakes', 'category': 'Data Engineering'}, {'canonical_skill': 'PostgreSQL', 'alias': 'postgresql', 'category': 'Databases'}, {'canonical_skill': 'PostgreSQL', 'alias': 'postgres', 'category': 'Databases'}, {'canonical_skill': 'MySQL', 'alias': 'mysql', 'category': 'Databases'}, {'canonical_skill': 'MongoDB', 'alias': 'mongodb', 'category': 'Databases'}, {'canonical_skill': 'Redis', 'alias': 'redis', 'category': 'Databases'}, {'canonical_skill': 'DynamoDB', 'alias': 'dynamodb', 'category': 'Databases'}, {'canonical_skill': 'BigQuery', 'alias': 'bigquery', 'category': 'Databases'}, {'canonical_skill': 'BigQuery', 'alias': 'big query', 'category': 'Databases'}, {'canonical_skill': 'Redshift', 'alias': 'redshift', 'category': 'Databases'}, {'canonical_skill': 'SQL Server', 'alias': 'sql server', 'category': 'Databases'}, {'canonical_skill': 'SQL Server', 'alias': 'microsoft sql server', 'category': 'Databases'}, {'canonical_skill': 'Oracle', 'alias': 'oracle database', 'category': 'Databases'}, {'canonical_skill': 'Cassandra', 'alias': 'cassandra', 'category': 'Databases'}, {'canonical_skill': 'Elasticsearch', 'alias': 'elasticsearch', 'category': 'Databases'}, {'canonical_skill': 'Elasticsearch', 'alias': 'elastic search', 'category': 'Databases'}, {'canonical_skill': 'Vector Databases', 'alias': 'vector database', 'category': 'Databases'}, {'canonical_skill': 'Vector Databases', 'alias': 'vector databases', 'category': 'Databases'}, {'canonical_skill': 'Vector Databases', 'alias': 'vector db', 'category': 'Databases'}, {'canonical_skill': 'Vector Databases', 'alias': 'vector dbs', 'category': 'Databases'}, {'canonical_skill': 'AWS', 'alias': 'aws', 'category': 'Cloud'}, {'canonical_skill': 'AWS', 'alias': 'amazon web services', 'category': 'Cloud'}, {'canonical_skill': 'GCP', 'alias': 'gcp', 'category': 'Cloud'}, {'canonical_skill': 'GCP', 'alias': 'google cloud platform', 'category': 'Cloud'}, {'canonical_skill': 'GCP', 'alias': 'google cloud', 'category': 'Cloud'}, {'canonical_skill': 'Azure', 'alias': 'microsoft azure', 'category': 'Cloud'}, {'canonical_skill': 'Azure', 'alias': 'azure', 'category': 'Cloud'}, {'canonical_skill': 'SageMaker', 'alias': 'sagemaker', 'category': 'Cloud'}, {'canonical_skill': 'SageMaker', 'alias': 'sage maker', 'category': 'Cloud'}, {'canonical_skill': 'Vertex AI', 'alias': 'vertex ai', 'category': 'Cloud'}, {'canonical_skill': 'AWS Lambda', 'alias': 'aws lambda', 'category': 'Cloud'}, {'canonical_skill': 'AWS Lambda', 'alias': 'lambda functions', 'category': 'Cloud'}, {'canonical_skill': 'Docker', 'alias': 'docker', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Kubernetes', 'alias': 'kubernetes', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Kubernetes', 'alias': 'k8s', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'MLflow', 'alias': 'mlflow', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'CI/CD', 'alias': 'ci/cd', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'CI/CD', 'alias': 'continuous integration', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'CI/CD', 'alias': 'continuous delivery', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'CI/CD', 'alias': 'continuous deployment', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Deployment', 'alias': 'model deployment', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Deployment', 'alias': 'deploy models', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Deployment', 'alias': 'deploying models', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Monitoring', 'alias': 'model monitoring', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Monitoring', 'alias': 'monitoring models', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Serving', 'alias': 'model serving', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Model Serving', 'alias': 'serving models', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Kubeflow', 'alias': 'kubeflow', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Terraform', 'alias': 'terraform', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'GitHub Actions', 'alias': 'github actions', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Jenkins', 'alias': 'jenkins', 'category': 'MLOps / Deployment'}, {'canonical_skill': 'Tableau', 'alias': 'tableau', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Power BI', 'alias': 'power bi', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Power BI', 'alias': 'powerbi', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Looker', 'alias': 'looker', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Matplotlib', 'alias': 'matplotlib', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Seaborn', 'alias': 'seaborn', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Plotly', 'alias': 'plotly', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Data Visualization', 'alias': 'data visualization', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Data Visualization', 'alias': 'data visualisation', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Dashboarding', 'alias': 'dashboarding', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Dashboarding', 'alias': 'dashboards', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Dashboarding', 'alias': 'dashboard', 'category': 'Data Visualisation / BI'}, {'canonical_skill': 'Git', 'alias': 'git', 'category': 'Development Tools'}, {'canonical_skill': 'GitHub', 'alias': 'github', 'category': 'Development Tools'}, {'canonical_skill': 'Linux', 'alias': 'linux', 'category': 'Development Tools'}, {'canonical_skill': 'Jupyter', 'alias': 'jupyter', 'category': 'Development Tools'}, {'canonical_skill': 'Jupyter', 'alias': 'jupyter notebook', 'category': 'Development Tools'}, {'canonical_skill': 'VS Code', 'alias': 'vs code', 'category': 'Development Tools'}, {'canonical_skill': 'VS Code', 'alias': 'visual studio code', 'category': 'Development Tools'}, {'canonical_skill': 'REST APIs', 'alias': 'rest api', 'category': 'Development Tools'}, {'canonical_skill': 'REST APIs', 'alias': 'rest apis', 'category': 'Development Tools'}, {'canonical_skill': 'REST APIs', 'alias': 'restful api', 'category': 'Development Tools'}, {'canonical_skill': 'REST APIs', 'alias': 'restful apis', 'category': 'Development Tools'}, {'canonical_skill': 'APIs', 'alias': 'apis', 'category': 'Development Tools'}, {'canonical_skill': 'APIs', 'alias': 'api development', 'category': 'Development Tools'}, {'canonical_skill': 'Microservices', 'alias': 'microservices', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Microservices', 'alias': 'microservice architecture', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Distributed Systems', 'alias': 'distributed systems', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Distributed Systems', 'alias': 'distributed system', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Cloud Computing', 'alias': 'cloud computing', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Data Governance', 'alias': 'data governance', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Data Quality', 'alias': 'data quality', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Data Modeling', 'alias': 'data modeling', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Data Modeling', 'alias': 'data modelling', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Knowledge Graphs', 'alias': 'knowledge graph', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Knowledge Graphs', 'alias': 'knowledge graphs', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Graph Neural Networks', 'alias': 'graph neural networks', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Graph Neural Networks', 'alias': 'graph neural network', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Graph Neural Networks', 'alias': 'gnn', 'category': 'Methods / Architecture'}, {'canonical_skill': 'Graph Neural Networks', 'alias': 'gnns', 'category': 'Methods / Architecture'}]

skill_dictionary = (
    pd.DataFrame(skill_records)
    .drop_duplicates()
    .sort_values(["category", "canonical_skill", "alias"])
    .reset_index(drop=True)
)

print("Canonical skills:", skill_dictionary["canonical_skill"].nunique())
print("Aliases:", len(skill_dictionary))
print("Categories:", skill_dictionary["category"].nunique())

skill_dictionary.head(20)

Canonical skills: 132
Aliases: 232
Categories: 12


,canonical_skill,alias,category
0,AWS,amazon web services,Cloud
1,AWS,aws,Cloud
2,AWS Lambda,aws lambda,Cloud
3,AWS Lambda,lambda functions,Cloud
4,Azure,azure,Cloud
5,Azure,microsoft azure,Cloud
6,GCP,gcp,Cloud
7,GCP,google cloud,Cloud
8,GCP,google cloud platform,Cloud
9,SageMaker,sage maker,Cloud


In [9]:
taxonomy_summary = (
    skill_dictionary
    .groupby("category")
    .agg(
        canonical_skills=("canonical_skill", "nunique"),
        aliases=("alias", "size")
    )
    .sort_values("canonical_skills", ascending=False)
)

taxonomy_summary

,canonical_skills,aliases
category,,
Data Engineering,17,29
Data Analysis & Statistics,14,23
NLP / Generative AI,14,34
Databases,12,19
ML Frameworks & Libraries,12,14
Programming,12,19
MLOps / Deployment,11,19
Machine Learning & AI,11,23
Data Visualisation / BI,8,12


## 3. Build alias-to-canonical normalisation rules

Each detected alias is converted to one canonical representation.  
This prevents equivalent expressions from being counted as separate skills.

In [11]:
alias_to_canonical = dict(
    zip(
        skill_dictionary["alias"].astype(str),
        skill_dictionary["canonical_skill"].astype(str)
    )
)

lower_alias_to_canonical = {
    k.lower(): v for k, v in alias_to_canonical.items()
}

def normalise_skill(skill: str) -> str:
    skill = str(skill).strip()

    if skill in alias_to_canonical:
        return alias_to_canonical[skill]

    return lower_alias_to_canonical.get(skill.lower(), skill)

examples = ["ML", "machine learning", "LLMs", "amazon web services", "PowerBI"]
for item in examples:
    print(f"{item!r} -> {normalise_skill(item)!r}")

'ML' -> 'ML'
'machine learning' -> 'Machine Learning'
'LLMs' -> 'Large Language Models'
'amazon web services' -> 'AWS'
'PowerBI' -> 'Power BI'


## 4. Safe matching rules for validation and frequency analysis

Naive substring matching can create false positives. For example, checking whether `"r"` occurs anywhere in a description would incorrectly match thousands of ordinary words.

For validation in this notebook:

- aliases are matched using alphanumeric boundaries;
- standalone programming language `R` is case-sensitive;
- ordinary aliases are case-insensitive;
- punctuation in aliases such as `C++` and `C#` is escaped safely.

Step 3 should still implement the production extractor with spaCy `PhraseMatcher`.

In [13]:
def alias_pattern(alias: str):
    alias = str(alias).strip()

    if alias == "R":
        return re.compile(r"(?<![A-Za-z0-9])R(?![A-Za-z0-9])")

    escaped = re.escape(alias)
    return re.compile(
        rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])",
        flags=re.IGNORECASE
    )

compiled_aliases = {
    alias: alias_pattern(alias)
    for alias in skill_dictionary["alias"].astype(str).unique()
}

def alias_present(text: str, alias: str) -> bool:
    if pd.isna(text):
        return False
    return bool(compiled_aliases[alias].search(str(text)))

boundary_tests = {
    ("We use R for statistical modelling.", "R"): True,
    ("We build recommendation systems.", "R"): False,
    ("Experience with C++ is required.", "c++"): True,
    ("Python and SQL are required.", "python"): True,
}

for (text, alias), expected in boundary_tests.items():
    result = alias_present(text, alias)
    print(alias, result, "expected:", expected)

R True expected: True
R False expected: False
c++ True expected: True
python True expected: True


## 5. Calculate overall skill frequencies

A canonical skill is counted at most once per job, even if it appears multiple times.

`matched_job_pct = jobs mentioning the skill / all jobs × 100`

In [15]:
descriptions = jobs["job_description"].fillna("").astype(str)

alias_job_indices = {}
for alias in skill_dictionary["alias"].astype(str).unique():
    pattern = compiled_aliases[alias]
    matched = {
        idx for idx, text in descriptions.items()
        if pattern.search(text)
    }
    alias_job_indices[alias] = matched

canonical_job_indices = {}
for canonical, group in skill_dictionary.groupby("canonical_skill"):
    combined = set()
    for alias in group["alias"].astype(str):
        combined.update(alias_job_indices[alias])
    canonical_job_indices[canonical] = combined

overall_counts = {
    canonical: len(indices)
    for canonical, indices in canonical_job_indices.items()
}

alias_counts = {
    alias: len(indices)
    for alias, indices in alias_job_indices.items()
}

skill_dictionary["matched_job_count"] = (
    skill_dictionary["canonical_skill"].map(overall_counts).astype(int)
)

skill_dictionary["matched_job_pct"] = (
    skill_dictionary["matched_job_count"] / len(jobs) * 100
).round(2)

skill_dictionary["alias_job_count"] = (
    skill_dictionary["alias"].map(alias_counts).astype(int)
)

overall_skill_frequency = (
    skill_dictionary[
        ["canonical_skill", "category", "matched_job_count", "matched_job_pct"]
    ]
    .drop_duplicates("canonical_skill")
    .sort_values(
        ["matched_job_count", "canonical_skill"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

overall_skill_frequency.head(25)

,canonical_skill,category,matched_job_count,matched_job_pct
0,Python,Programming,456,66.28
1,Large Language Models,NLP / Generative AI,321,46.66
2,Machine Learning,Machine Learning & AI,292,42.44
3,SQL,Programming,239,34.74
4,APIs,Development Tools,172,25.00
5,Statistics,Data Analysis & Statistics,139,20.20
6,Data Pipelines,Data Engineering,134,19.48
7,Distributed Systems,Methods / Architecture,124,18.02
8,Artificial Intelligence,Machine Learning & AI,123,17.88
9,Apache Spark,Data Engineering,122,17.73


## 6. Calculate skill frequency by career category

For each career category, the percentage represents the share of advertisements in that career mentioning the canonical skill.

These values can later be used to build career skill profiles.

In [17]:
career_rows = []

for career, career_df in jobs.groupby("career_category"):
    career_indices = set(career_df.index)
    n_jobs = len(career_df)

    for canonical, matched_indices in canonical_job_indices.items():
        count = len(career_indices.intersection(matched_indices))

        category = (
            skill_dictionary.loc[
                skill_dictionary["canonical_skill"] == canonical,
                "category"
            ].iloc[0]
        )

        career_rows.append({
            "career_category": career,
            "canonical_skill": canonical,
            "skill_category": category,
            "job_count": count,
            "career_job_total": n_jobs,
            "job_pct": round(count / n_jobs * 100, 2) if n_jobs else 0.0
        })

skill_frequency_by_career = pd.DataFrame(career_rows)

skill_frequency_by_career = skill_frequency_by_career.sort_values(
    ["career_category", "job_pct", "canonical_skill"],
    ascending=[True, False, True]
).reset_index(drop=True)

skill_frequency_by_career.head(30)

,career_category,canonical_skill,skill_category,job_count,career_job_total,job_pct
0,AI Engineer,Large Language Models,NLP / Generative AI,165,275,60.00
1,AI Engineer,Python,Programming,152,275,55.27
2,AI Engineer,APIs,Development Tools,109,275,39.64
3,AI Engineer,Distributed Systems,Methods / Architecture,71,275,25.82
4,AI Engineer,TypeScript,Programming,69,275,25.09
5,AI Engineer,Kubernetes,MLOps / Deployment,57,275,20.73
6,AI Engineer,CI/CD,MLOps / Deployment,52,275,18.91
7,AI Engineer,Machine Learning,Machine Learning & AI,52,275,18.91
8,AI Engineer,AWS,Cloud,48,275,17.45
9,AI Engineer,Generative AI,Machine Learning & AI,46,275,16.73


In [18]:
top_skills_by_career = (
    skill_frequency_by_career
    .sort_values(["career_category", "job_pct"], ascending=[True, False])
    .groupby("career_category", group_keys=False)
    .head(15)
)

for career, group in top_skills_by_career.groupby("career_category"):
    print(f"\n=== {career} ===")
    print(
        group[["canonical_skill", "job_count", "job_pct"]]
        .to_string(index=False)
    )


=== AI Engineer ===
        canonical_skill  job_count  job_pct
  Large Language Models        165    60.00
                 Python        152    55.27
                   APIs        109    39.64
    Distributed Systems         71    25.82
             TypeScript         69    25.09
             Kubernetes         57    20.73
                  CI/CD         52    18.91
       Machine Learning         52    18.91
                    AWS         48    17.45
          Generative AI         46    16.73
Artificial Intelligence         45    16.36
            Fine-tuning         42    15.27
             JavaScript         42    15.27
                    GCP         34    12.36
         Data Pipelines         33    12.00

=== Data Analyst ===
   canonical_skill  job_count  job_pct
               SQL         26    96.30
      Dashboarding         23    85.19
            Python         22    81.48
           Tableau         16    59.26
               dbt         12    44.44
      Data Quality 

## 7. Basic dictionary validation

Checks:

- aliases mapped to more than one canonical skill;
- canonical skills with zero matches;
- short aliases that require extra care in phrase matching.

In [20]:
duplicate_alias_mapping = (
    skill_dictionary
    .groupby("alias")["canonical_skill"]
    .nunique()
    .sort_values(ascending=False)
)

conflicting_aliases = duplicate_alias_mapping[duplicate_alias_mapping > 1]

zero_match_skills = overall_skill_frequency[
    overall_skill_frequency["matched_job_count"] == 0
]

short_aliases = (
    skill_dictionary[
        skill_dictionary["alias"].astype(str).str.len() <= 3
    ][["canonical_skill", "alias", "category"]]
    .sort_values(["canonical_skill", "alias"])
)

print("Aliases mapped to multiple canonical skills:", len(conflicting_aliases))
if len(conflicting_aliases):
    print(conflicting_aliases)

print("\nCanonical skills with zero matches:", len(zero_match_skills))
print(zero_match_skills.to_string(index=False))

print("\nShort aliases requiring careful matching:")
print(short_aliases.to_string(index=False))

Aliases mapped to multiple canonical skills: 0

Canonical skills with zero matches: 7
canonical_skill                   category  matched_job_count  matched_job_pct
     AWS Lambda                      Cloud                  0              0.0
      Cassandra                  Databases                  0              0.0
  Data Cleaning Data Analysis & Statistics                  0              0.0
           NLTK  ML Frameworks & Libraries                  0              0.0
         Oracle                  Databases                  0              0.0
          SciPy  ML Frameworks & Libraries                  0              0.0
          spaCy  ML Frameworks & Libraries                  0              0.0

Short aliases requiring careful matching:
            canonical_skill alias                   category
                        AWS   aws                      Cloud
                         C#    c#                Programming
                        C++   c++                Program

## 8. Export Step 2 outputs

The handover package for Step 3 contains:

- `skills_dictionary.csv`
- `skill_frequency_by_career.csv`
- `normalisation.py`
- `skill_taxonomy.md`

In [22]:
skills_output = OUTPUT_DIR / "skills_dictionary.csv"
career_output = OUTPUT_DIR / "skill_frequency_by_career.csv"
normalisation_output = OUTPUT_DIR / "normalisation.py"
taxonomy_output = OUTPUT_DIR / "skill_taxonomy.md"

skill_dictionary.to_csv(
    skills_output,
    index=False,
    encoding="utf-8-sig"
)

skill_frequency_by_career.to_csv(
    career_output,
    index=False,
    encoding="utf-8-sig"
)

mapping_repr = repr(alias_to_canonical)

normalisation_lines = [
    '"""Skill normalisation rules generated in Step 2."""',
    "",
    f"ALIAS_TO_CANONICAL = {mapping_repr}",
    "",
    "LOWER_ALIAS_TO_CANONICAL = {k.lower(): v for k, v in ALIAS_TO_CANONICAL.items()}",
    "",
    "def normalise_skill(skill: str) -> str:",
    "    skill = str(skill).strip()",
    "    if skill in ALIAS_TO_CANONICAL:",
    "        return ALIAS_TO_CANONICAL[skill]",
    "    return LOWER_ALIAS_TO_CANONICAL.get(skill.lower(), skill)",
]

normalisation_output.write_text(
    "\n".join(normalisation_lines),
    encoding="utf-8"
)

taxonomy_lines = [
    "# Skill Taxonomy — Step 2",
    "",
    "This taxonomy supports explainable skill extraction from U.S. Data/AI job advertisements "
    "and Australian students' study materials.",
    "",
    "## Summary",
    f"- Canonical skills: {skill_dictionary['canonical_skill'].nunique()}",
    f"- Aliases: {len(skill_dictionary)}",
    f"- Categories: {skill_dictionary['category'].nunique()}",
    "",
    "## Categories",
    "",
]

for category, group in skill_dictionary.groupby("category", sort=True):
    canonical = sorted(group["canonical_skill"].unique())
    taxonomy_lines.append(f"### {category}")
    taxonomy_lines.append(", ".join(canonical))
    taxonomy_lines.append("")

taxonomy_lines.extend([
    "## Normalisation principle",
    "",
    "Aliases are mapped to a single canonical representation.",
    "",
    "Short or ambiguous aliases such as `R` require strict token or phrase matching. "
    "Step 3 should use spaCy PhraseMatcher rather than arbitrary substring matching.",
    "",
    "## Interpretation",
    "",
    "Skill frequencies represent mentions in job advertisements. "
    "They are not measures of candidate mastery, employability, or hiring probability.",
])

taxonomy_output.write_text(
    "\n".join(taxonomy_lines),
    encoding="utf-8"
)

print("Exported:")
for p in [
    skills_output,
    career_output,
    normalisation_output,
    taxonomy_output
]:
    print(" -", p)

Exported:
 - outputs\skills_dictionary.csv
 - outputs\skill_frequency_by_career.csv
 - outputs\normalisation.py
 - outputs\skill_taxonomy.md


## 9. Step 3 handover

Step 3 should use:

1. `job_cleaned.csv`
2. `outputs/skills_dictionary.csv`
3. `outputs/normalisation.py`

Recommended pipeline:

`job_description → spaCy PhraseMatcher → matched alias → canonical skill → deduplicate per job`

Expected output:

| job_id | career_category | job_title | extracted_skills |
|---|---|---|---|
| ... | Data Scientist | ... | Python, SQL, Machine Learning, AWS |

The same extraction logic can later be applied to Australian university study materials so the two skill profiles are comparable.